In [1]:
#Offsetting the SSH and (U,V) or SST by 1 snapshot (48 hours) in both training and testing data, and test the performance. 
#We realize this with minimal changes to previous codes, by just shifting all fields that are not SSH by 1. 
#In this code, SSH are BEHIND (U,V) or SST in time.
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import r2_score as R2
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score as r2
from copy import deepcopy
import utils
from unet import UNet_nobatchnorm
from scipy.stats import pearsonr
#JU's addtion to automate inputs and outputs
import helper_functions as hf
import os
def save_fn(var_input_list, var_output_list):
    var_input_join  = '_and_'.join(var_input_list)
    var_output_join = '_and_'.join(var_output_list)
    return '{}_to_{}'.format(var_input_join, var_output_join)

torch.cuda.set_device(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print ('Running on ', device)

Running on  cuda:0


In [2]:
print(torch.__version__)
print(torch.version.cuda)

2.5.1
12.6


In [3]:
maxEpochs =  300#small number is taken for debugging
nensemble = 10 #How many training sessions are run for each configuration 
Nbase = 16

In [4]:
!nvidia-smi #GPU usage should be maxed out during training; tune batch_size according to that

Sat Feb 21 11:42:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:03:00.0 Off |                    0 |
| N/A   42C    P0             57W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
root_dir = '/work/uo0780/u241359/project_tide_synergy/data/'
nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)


Ntrain_fromfiles = np.sum([nc.dimensions['time_counter'].size for nc in nctrains], axis = 0)
Ntest_fromfiles = np.sum([nc.dimensions['time_counter'].size for nc in nctest], axis = 0)

print('number of training records:', Ntrain_fromfiles)
print('number of testing records:', Ntest_fromfiles)

numTrainFiles = len(nctrains)
numRecsFile = nctrains[0].dimensions['time_counter'].size #How many snapshots in time in each data set there is
print (numRecsFile)


def preload_data(nctrains, total_records):
    #total_records = Ntrain#sum(nc.dimensions['time_counter'].size for nc in nctrains)
    #dimensions of data of the nc file.
    max_height = 722
    max_width = 258
    all_input_data = np.zeros((total_records, N_inp, max_height, max_width))*np.nan
    all_output_data = np.zeros((total_records, N_out, max_height, max_width))*np.nan
    current_index = 0
    for ncindex, ncdata in enumerate(nctrains):
        num_recs = ncdata.dimensions['time_counter'].size
        rec_slice = slice(current_index, current_index + num_recs)
        
        for ind, var_name in enumerate(var_input_names):
            data_slice = np.squeeze(ncdata.variables[var_name])
            # print('data_slice shape:')
            # print(data_slice.shape)        
            #all_input_data[rec_slice, ind, :, :] = data_slice
            #For some variables, the dimensions in (x, y) may be smaller than (max_height, max_width). Changing the code so that it adapts them.
            # Get the actual dimensions of data_slice
            slice_height, slice_width = data_slice.shape[-2], data_slice.shape[-1]
            # Place data_slice into the corresponding slice of all_input_data
            all_input_data[rec_slice, ind, :slice_height, :slice_width] = data_slice
    

        for ind, var_name in enumerate(var_output_names):
            data_slice = np.squeeze(ncdata.variables[var_name])
            #all_output_data[rec_slice, ind, :, :] = data_slice
            # Get the actual dimensions of data_slice
            slice_height, slice_width = data_slice.shape[-2], data_slice.shape[-1]
            # Place data_slice into the corresponding slice of all_input_data
            all_output_data[rec_slice, ind, :slice_height, :slice_width] = data_slice

        current_index += num_recs
        
    return all_input_data, all_output_data
    
# Modify the loadtrain function to pull data from preloaded memory
def loaddata_preloaded_train(index, batch_size, all_input_data, all_output_data):
    rec_slice = slice(index, index + batch_size)
    lim = 720
    width = 256
    yslice = slice(0, lim)
    xslice = slice(0, width)
    # print('rec_slice is:')
    # print(rec_slice)
    # print('mean of squared values of loaded input data:')
    # print("{0:0.32f}".format(np.nanmean(all_input_data[rec_slice, :, yslice, xslice]**2)))
    return (all_input_data[rec_slice, :, yslice, xslice], 
            all_output_data[rec_slice, :, yslice, xslice])
#Load test data as one single batch
def loaddata_preloaded_test(all_input_data, all_output_data):
    #rec_slice = slice(index, index + batch_size)
    lim = 720
    width = 256
    yslice = slice(0, lim)
    xslice = slice(0, width)
    # print('rec_slice is:')
    # print(rec_slice)
    # print('mean of squared values of loaded input data:')
    # print("{0:0.32f}".format(np.nanmean(all_input_data[rec_slice, :, yslice, xslice]**2)))
    return (all_input_data[:, :, yslice, xslice], 
            all_output_data[:, :, yslice, xslice])


def load_variable(ncdata, ncindex, variable, rec_slice, yslice, xslice):
    data_squeezed = np.squeeze(ncdata[ncindex].variables[variable])
    return data_squeezed[rec_slice, yslice, xslice]

def apply_fixed_offset_ssh_trails_by_segments(all_input, all_output, shifted_input_indices, seg_lengths, lag=1):
    """
    SSH trails by 1 snapshot relative to selected other input channels, within each segment (.nc file).
    We keep SSH and outputs at time t, but use shifted channels from time t-1.

    Concretely (lag=1):
      x_new[t, all channels] <- all_input[t]  (base is SSH time)
      x_new[t, shifted channels] <- all_input[t-1, shifted channels]
      y_new[t] <- all_output[t]
    To avoid cyclic pairing, we DROP the first snapshot of each segment.

    all_input:  (T, Cin, H, W)
    all_output: (T, Cout, H, W)
    seg_lengths: e.g. [150,150,150,150]
    """
    assert lag == 1, "Written for lag=1 (easy to generalize later)."
    shifted_input_indices = list(shifted_input_indices)

    T, Cin, H, W = all_input.shape
    Tout, Cout = all_output.shape[0], all_output.shape[1]
    assert Tout == T, "Input/output time dimension mismatch."

    new_T = sum(L - lag for L in seg_lengths)  # drop first lag per segment
    x_new = np.zeros((new_T, Cin, H, W), dtype=all_input.dtype)
    y_new = np.zeros((new_T, Cout, H, W), dtype=all_output.dtype)

    in_pos = 0
    out_pos = 0

    for L in seg_lengths:
        if L <= lag:
            raise ValueError(f"Segment length {L} too short for lag={lag}.")

        s0 = in_pos
        s1 = in_pos + L
        keep = L - lag

        # Base alignment is SSH time t = s0+lag ... s1-1 (drop first snapshot)
        x_new[out_pos:out_pos+keep, :, :, :] = all_input[s0+lag:s1, :, :, :]

        # Overwrite shifted channels with t-lag = s0 ... s1-lag-1
        x_new[out_pos:out_pos+keep, shifted_input_indices, :, :] = all_input[s0:s1-lag, shifted_input_indices, :, :]

        # Outputs aligned with SSH time (same as base): drop first snapshot
        y_new[out_pos:out_pos+keep, :, :, :] = all_output[s0+lag:s1, :, :, :]

        in_pos += L
        out_pos += keep

    return x_new, y_new

def pad_to_multiple_by_repeating_last(all_input, all_output, batch_size):
    """
    Pads the *end* of the dataset by repeating the last snapshot. This is so that the number of snapshots remain unchanged after calling apply_fixed_offset_ssh_leads. 
    The reason why I want to do that is because I want the number of snapshots to be divisible by batch sizes during training. It artifically repeats the last snapshot, 
    but that is fine, as that is just one snapshot each simulation 150 snapshots).
    """
    N = all_input.shape[0]
    r = N % batch_size #If r == 0, then N is already an exact multiple of batch_size. In that case, you don’t need any padding, so the function just returns the arrays unchanged:
    if r == 0:
        return all_input, all_output

    n_pad = batch_size - r
    inp_pad = np.repeat(all_input[-1:, ...], n_pad, axis=0)
    out_pad = np.repeat(all_output[-1:, ...], n_pad, axis=0)

    all_input  = np.concatenate([all_input,  inp_pad], axis=0)
    all_output = np.concatenate([all_output, out_pad], axis=0)
    return all_input, all_output


number of training records: 600
number of testing records: 150
150


In [6]:
def run_model(var_input_names, var_output_names, save_fn_prefix, N_inp, N_out, iensemble, R2_all, corr_all):
    ytest_slice = slice(0, 720)
    xtest_slice = slice(0, 256)
    rectest_slice = slice(0, 150)

    def totorch(x):
        return torch.tensor(x, dtype = torch.float).cuda()

    model = UNet_nobatchnorm(N_inp, N_out, bilinear = True, Nbase = Nbase).cuda()
    #model = torch.compile(UNet(N_inp, N_out, bilinear = True, Nbase = Nbase).cuda())

    if iensemble == 0:
        input = torch.randn(1,N_inp,256,720).to(device) 
        output = model(input)
        print('Model has ', utils.nparams(model)/1e6, ' million params')

    # for index in range(0, Ntrain, batch_size):
    #     inp, out = loadtrain_preloaded(index, batch_size, all_train_input, all_train_output)
    #     print(inp.shape, out.shape)
#         print(np.nanmean(inp**2), np.max(inp**2), inp.shape)
#         print(np.nanmean(out**2), np.max(out**2), inp.shape)

    inp_test, out_test = loaddata_preloaded_test(all_test_input, all_test_output)
    #inp, out_test = loadtest()
    # print('shapes of input and output TEST data:')
    # print(inp_test.shape, out_test.shape)
    with torch.no_grad():
        inp_test = totorch(inp_test)

    Tcycle = 10
    criterion_train  = nn.L1Loss()
    optim = torch.optim.AdamW(model.parameters(), lr=lr0, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-5*100) #increase weight_decay ***

    r2_test = np.zeros(maxEpochs)
    epochmin = []
    maxr2l = []

    learn = np.zeros(maxEpochs)
    minloss = 1000
    maxR2 = -1000
    minlosscount = 0
    perm = False

    model_best = deepcopy(model)  # Initialize before the loop for safety

    #print('Starting training loop')
    for epoch in tqdm(range(maxEpochs)):
        lr = utils.cosineSGDR(optim, epoch, T0=Tcycle, eta_min=0, eta_max=lr0, scheme = 'constant')  #captioning this seems to make the printed corr larger??***
        model.train()
        index_perm = np.arange(0, Ntrain, batch_size)
        
        if perm:
            index_perm = np.random.permutation(index_perm)
        
        for index in index_perm:
            inp, out = loaddata_preloaded_train(index, batch_size, all_train_input, all_train_output)            
#           inp, out = loadtrain(index, batch_size, nctrains)
            inp, out = totorch(inp), totorch(out)
            #continue #do this to pause the later operations to check how long it takes for the steps up to this 
            out_mod = model(inp)
            loss = criterion_train(out.squeeze(), out_mod.squeeze())
            #Set gradient to zero
            optim.zero_grad()
            #Compute gradients       
            loss.backward()
            #Update parameters with new gradient
            optim.step()
            #Record train loss
            #scheduler.step()
          
        model.eval()
        with torch.no_grad():
            #model_cpu = model.to('cpu')
            #out_mod = model_cpu(inp_test.to('cpu'))
            out_mod=model(inp_test)
            
            r2 = R2(out_test.flatten(), (out_mod).cpu().numpy().flatten())
            r2_test[epoch] = r2
            #print('Debugging: R2 of current epoch:', r2)#Debugging
            #record current best model and best predictions
            if maxR2 <  r2:
                maxR2 = r2
                epochmin.append(epoch)
                maxr2l.append(maxR2)                
                model_best = deepcopy(model)
                corr, pval = pearsonr(out_test.flatten(), (out_mod).cpu().numpy().flatten())
                print('R2:', r2, ' corr: ', corr, ' pval: ', pval)
            #model = model_cpu.to(device)

    #_, out_test = loadtest()
    model_best.eval()
    with torch.no_grad():
    #     inp_test = totorch(inp)
        model_best.to('cpu') #added by HW 
        out_mod = model_best(inp_test.to('cpu')).detach().cpu().numpy()

    R2_all[iensemble]=R2(out_test.flatten(), out_mod.flatten())
    corr_all[iensemble]=pearsonr(out_test.flatten(), out_mod.flatten())[0]
    # print('Best model R2:', R2_all[iensemble])#pearsonr(out_test.flatten(), out_mod.flatten())[0])
    # print('Best model corr:', corr_all[iensemble])#R2(out_test.flatten(), out_mod.flatten()))

    # Nx, Ny = out_test.shape[2:]; Nx, Ny

    # print(out_mod.shape, 'outout model shape')
    dr = '/work/uo0780/u241359/project_tide_synergy/trainedmodels' #'./models/to_vel'
    os.makedirs(dr, exist_ok=True) # exist_ok=True allows the function to do nothing (i.e., not raise an error) if the directory already exists.
    fstr = f'{save_fn_prefix}_rp_{iensemble}'
    PATH = dr + f'/{fstr}.pth'
    torch.save(model_best.state_dict(), PATH)

In [ ]:
vi1 = 'ssh_ins'
vi2 = 'u_xy_ins'
vi3 = 'v_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'
#Important: shifted_input_indices a few lines later change in each config. in later codes processing the trained data, remember to use apply_fixed_offset_ssh_trails_by_segments consistent with the lines below
#to make sure that shifted_input_indices are consistent. 

save_fn_prefix  = 'any_{}{}{}_{}{}_nobatchnorm_offset_48hr_ssh_trails'.format(vi1, vi2, vi3, vo1, vo2)

batch_size = 60 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.

var_input_names = [vi1, vi2, vi3]
var_output_names = [vo1, vo2]

N_inp = len(var_input_names)
N_out = len(var_output_names)

lr0 = 0.005*10/batch_size #Roughly should scale inversely to batch_size

#Recording performance metrics on test data after eaching training cycle
R2_all = np.zeros(nensemble)
corr_all = np.zeros(nensemble)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)


# Add offset, using the ordering in var_input_names:
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']
# segment lengths per .nc file (train: 4 sims, test: 1 sim)
seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)


all_train_input, all_train_output = apply_fixed_offset_ssh_trails_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1,2], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_trails_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1,2], seg_lengths=seg_test, lag=1
)


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


for iensemble in np.arange(nensemble):
    run_model(var_input_names, var_output_names, save_fn_prefix, N_inp, N_out, iensemble, R2_all, corr_all)  
print('R2 from the best models in each run are:')
print(R2_all)
print('corr from the best models in each run are:')
print(corr_all)

mean and variance of all input data:
[ 0.03322337  0.03565685 -0.00190692] [0.31172642 0.04678398 0.04873947]
mean and variance of all output data:
[-5.19675373e-04 -9.81703693e-05] [9.42781426e-05 1.02080678e-04]
number of training records (after offset and before padding): 596
number of testing records (after offset): 149
train input shape:
(596, 3, 722, 258)
number of training records (after offset and padding): 600
number of testing records (after offset): 149
Model has  1.124706  million params


  0%|          | 1/300 [00:07<39:36,  7.95s/it]

R2: -0.015439339241188677  corr:  0.029582322510007555  pval:  0.0


  1%|          | 2/300 [00:15<37:49,  7.61s/it]

R2: 0.010553900762471091  corr:  0.15677985991575802  pval:  0.0


...

 37%|███▋      | 111/300 [10:59<21:18,  6.76s/it]

R2: 0.7299921751707761  corr:  0.8549591060040681  pval:  0.0


 37%|███▋      | 112/300 [11:06<22:09,  7.07s/it]

R2: 0.745466170054051  corr:  0.8657970450107996  pval:  0.0


 41%|████      | 122/300 [12:02<16:41,  5.63s/it]

In [ ]:
vi1 = 'ssh_ins'
vi2 = 'T_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'
#Important: shifted_input_indices a few lines later change in each config. in later codes processing the trained data, remember to use apply_fixed_offset_ssh_trails_by_segments consistent with the lines below
#to make sure that shifted_input_indices are consistent. 


save_fn_prefix  = 'any_{}{}_{}{}_nobatchnorm_offset_48hr_ssh_trails'.format(vi1, vi2, vo1, vo2)
var_input_names = [vi1, vi2]
var_output_names = [vo1, vo2]

batch_size = 80 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.

N_inp = len(var_input_names)
N_out = len(var_output_names)

lr0 = 0.005*10/batch_size #Roughly should scale inversely to batch_size

#Recording performance metrics on test data after eaching training cycle
R2_all = np.zeros(nensemble)
corr_all = np.zeros(nensemble)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)


#Add offset
all_train_input, all_train_output = apply_fixed_offset_ssh_trails_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_trails_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1], seg_lengths=seg_test, lag=1
)


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


for iensemble in np.arange(nensemble):
    run_model(var_input_names, var_output_names, save_fn_prefix, N_inp, N_out, iensemble, R2_all, corr_all)  
print('R2 from the best models in each run are:')
print(R2_all)
print('corr from the best models in each run are:')
print(corr_all)